# News label quality check, sample 2 (news-sentiment track)

Second quality-check sample for the revised labeling prompt: 25 articles, 5 per source, seed 1, excluding the 25 articles of the first sample (`qc_sample.csv`).

- **Stage 1:** draw the sample and write `data/news/qc_sample_2.csv` with empty `my_*` columns for the hand labels. No LLM calls; no model labels are shown.
- **Stage 2:** label the sample twice with the pinned model and prompt `affefb1b22ff` (12 s spacing), writing only to `data/news/qc_labels.csv`, and compare with the hand labels.

## Step 0: Imports and settings

In [ ]:
from pathlib import Path

import pandas as pd

from cryptosense.news.qc import draw_qc_sample, write_qc_sample

DATA_DIR = Path.cwd().parent / "data"
NEWS_DIR = DATA_DIR / "news"
PER_SOURCE = 5
SEED = 1
FIRST_SAMPLE_PATH = NEWS_DIR / "qc_sample.csv"
QC_SAMPLE_2_PATH = NEWS_DIR / "qc_sample_2.csv"

## Step 1: Draw sample 2 and write it for hand labeling

The sample is drawn only if `qc_sample_2.csv` does not exist yet; `write_qc_sample` also refuses to overwrite it, so hand labels are never replaced by re-running this cell.

In [ ]:
first = pd.read_csv(FIRST_SAMPLE_PATH, dtype=str, keep_default_na=False)
if QC_SAMPLE_2_PATH.exists():
    print(f"{QC_SAMPLE_2_PATH.name} exists; sample not redrawn")
else:
    articles = pd.read_csv(NEWS_DIR / "articles.csv", dtype=str, keep_default_na=False)
    pool = articles[~articles["id"].isin(first["article_id"])]
    print(f"articles in file: {len(articles)}; excluded (sample 1): {len(articles) - len(pool)}")
    print("pool per source:", pool["source"].value_counts().sort_index().to_dict())
    write_qc_sample(draw_qc_sample(articles, per_source=PER_SOURCE, seed=SEED,
                                   exclude=first["article_id"]), QC_SAMPLE_2_PATH)

sample_2 = pd.read_csv(QC_SAMPLE_2_PATH, dtype=str, keep_default_na=False)
print(f"sampled: {len(sample_2)}")
print(sample_2["source"].value_counts().sort_index().to_string())
print("overlap with sample 1:", len(set(sample_2["article_id"]) & set(first["article_id"])))
print("columns:", list(sample_2.columns))
print("my_ columns all empty:", bool((sample_2.filter(like="my_") == "").all().all()))

## Stage 2: model labels and comparison with hand labels

## Step 2: Label the sample twice

Pinned model and prompt `affefb1b22ff`, temperature 0, 12 s between requests. Writes only to `qc_labels.csv` (never `labels.csv`). The cell stops before any request if the prompt version in `label.py` is not `affefb1b22ff`. Rows already present for the same run, article, model and prompt version are skipped, so re-running this cell does not label again; if the API fails, labeling stops and the next run resumes. Token totals cover the requests made by this call.

Only rows with prompt version `affefb1b22ff` (and the pinned model) for the sample-2 articles are compared below. Articles with every `my_` column empty are excluded from the comparison with hand labels but kept for run 1 versus run 2.

In [ ]:
from cryptosense.news.label import MODEL, PROMPT_VERSION, TOKEN_FIELDS, read_env_key
from cryptosense.news.qc import (
    agreement, current_model_labels, disagreements, label_qc_sample, merge_hand_and_model,
    run_consistency, split_hand_labeled,
)

EXPECTED_PROMPT_VERSION = "affefb1b22ff"
QC_LABELS_PATH = NEWS_DIR / "qc_labels.csv"
QC_BACKOFF_S = 12  # seconds between requests

if PROMPT_VERSION != EXPECTED_PROMPT_VERSION:
    raise RuntimeError(f"prompt_version is {PROMPT_VERSION}, expected {EXPECTED_PROMPT_VERSION}; not labeling")
result = label_qc_sample(read_env_key(), sample_2, QC_LABELS_PATH, runs=2, backoff_s=QC_BACKOFF_S)
print(f"model {MODEL}, prompt_version {PROMPT_VERSION}")
print(result)
print("token totals:", {field: result[field] for field in TOKEN_FIELDS})

labels = pd.read_csv(QC_LABELS_PATH, dtype=str, keep_default_na=False)
print("all rows by (model_requested, prompt_version):",
      labels.groupby(["model_requested", "prompt_version"]).size().to_dict())
in_sample = labels[labels["article_id"].isin(sample_2["article_id"])]
current = in_sample[(in_sample["model_requested"] == MODEL) & (in_sample["prompt_version"] == PROMPT_VERSION)]
print(current.groupby("run").agg(rows=("article_id", "size"),
                                 invalid=("invalid_reason", lambda s: (s != "").sum())).to_string())
print("model_reported:", current["model_reported"].value_counts().to_dict())
labels_2 = current_model_labels(in_sample)
print("valid rows compared:", len(labels_2))

In [ ]:
# Step 3: hand labels versus run 1 (articles without any hand label are excluded)
hand_labeled, excluded = split_hand_labeled(sample_2)
print(f"excluded (all my_ columns empty): {len(excluded)}")
for row in excluded.itertuples(index=False):
    print(f"  {row.source}: {row.title}")

merged = merge_hand_and_model(hand_labeled, labels_2, run=1)
pd.set_option("display.width", 200)
print(f"\narticles compared: {len(merged)}\n")
print("All sources")
print(agreement(merged).round(3).to_string())
is_coindesk = merged["source"] == "coindesk"
print("\nCoinDesk (title only)")
print(agreement(merged[is_coindesk]).round(3).to_string())
print("\nAll other sources")
print(agreement(merged[~is_coindesk]).round(3).to_string())

# Step 4: run 1 versus run 2 (all sample-2 articles), exact identity and sentiment direction
print("\nRun 1 versus run 2")
print(run_consistency(labels_2, 1, 2).round(3).to_string())

# Step 5: every disagreement (hand versus run 1)
found = disagreements(merged)
print(f"\ndisagreements: {len(found)}\n")
for i, row in enumerate(found.itertuples(index=False), 1):
    print(f"{i}. [{row.field}] {row.source}: {row.title}")
    print(f"   summary: {row.summary}")
    print(f"   hand: {row.hand}   model: {row.model}\n")

## Decision

**Decision (2026-10-25):** Accepted with a documented limitation. Passed: relevance agreement 88-92%, sentiment within one step 95-100%, run-to-run consistency 92-100% for relevance and sentiment. Not passed: sentiment direction agreement (market 30%, BTC 67%; bar 75%), mainly because the model rates ordinary positive news +1 where the hand labels used 0, as the written rule asks. Remaining model errors: analysis pieces and multi-story newsletters rated +1 instead of 0; these are handled in Phase 3 by rule. Prompt affefb1b22ff is frozen for the whole study.